In [1]:
from collections import defaultdict
from collections import Counter

# Caminho do arquivo .dic
dic_path = "./Dicionário/v2_SocialLIWC_formatado_ordenado.dic"


In [2]:
# 1) Mapear ID da categoria -> nome
id_to_category = {}

# 2) Contar palavras por ID de categoria
category_words = defaultdict(set)

with open(dic_path, "r", encoding="utf-8") as f:
    lines = f.readlines()

reading_categories = False

for line in lines:
    line = line.strip()

    # Ignorar linhas vazias
    if not line:
        continue

    # Detectar início e fim do bloco de categorias
    if line == "%":
        reading_categories = not reading_categories
        continue

    # ------------- BLOCO DE CATEGORIAS -------------
    if reading_categories:
        parts = line.split("\t")
        if len(parts) >= 2:
            cat_id = parts[0]
            cat_name = parts[1]
            id_to_category[cat_id] = cat_name
        continue

    # ------------- BLOCO DE PALAVRAS -------------
    parts = line.split("\t")

    word = parts[0]
    category_ids = parts[1:]

    for cat_id in category_ids:
        category_words[cat_id].add(word)

# 3) Exibir resultados ordenados pelo ID
print("Quantidade de palavras por categoria:\n")

for cat_id in sorted(category_words.keys(), key=int):
    cat_name = id_to_category.get(cat_id, "Categoria desconhecida")
    count = len(category_words[cat_id])
    print(f"{cat_id} - {cat_name}: {count} palavras")

Quantidade de palavras por categoria:

1 - Capacitismo: 150 palavras
2 - Gordofobia: 90 palavras
3 - Int.Religiosa: 152 palavras
4 - LGBTFobia: 67 palavras
5 - Misoginia: 141 palavras
6 - Xenofobia: 65 palavras
7 - Racismo: 81 palavras
8 - Etarismo: 58 palavras
9 - Prec.Politico: 83 palavras


In [3]:
# Contando se uma palavra pertence a 1 ou mais categorias.

contagem = Counter()

with open(dic_path, "r", encoding="utf-8") as f:
    for linha in f:
        linha = linha.strip()

        # Ignorar comentários e linhas vazias
        if not linha or linha.startswith("%"):
            continue

        partes = linha.split("\t")

        # Se tiver pelo menos palavra + 1 categoria
        if len(partes) >= 2:
            categorias = partes[1:]
            n_categorias = len(categorias)
            contagem[n_categorias] += 1

for n_cat in sorted(contagem):
    print(f"{contagem[n_cat]} palavras pertencem a {n_cat} categoria(s)")


806 palavras pertencem a 1 categoria(s)
45 palavras pertencem a 2 categoria(s)


In [4]:


# Mapear palavra -> categorias
word_to_categories = defaultdict(list)

with open(dic_path, "r", encoding="utf-8") as f:
    reading_categories = False

    for line in f:
        line = line.strip()

        if not line:
            continue

        if line == "%":
            reading_categories = not reading_categories
            continue

        # Ignorar bloco de categorias
        if reading_categories:
            continue

        parts = line.split("\t")

        if len(parts) >= 2:
            word = parts[0]
            category_ids = parts[1:]

            word_to_categories[word].extend(category_ids)

# 🔎 Filtrar palavras com mais de uma categoria
multi_category_words = {
    word: cats for word, cats in word_to_categories.items() if len(set(cats)) > 1
}

# Exibir resultado
print(f"\nPalavras que pertencem a mais de uma categoria: {len(multi_category_words)}\n")

for word, cats in multi_category_words.items():
    cat_names = [id_to_category.get(cat, cat) for cat in set(cats)]
    print(f"{word}: {cat_names}")


Palavras que pertencem a mais de uma categoria: 45

analfabeto: ['Capacitismo', 'Xenofobia']
atrasada: ['Capacitismo', 'Xenofobia']
atrasadas: ['Capacitismo', 'Xenofobia']
atrasado: ['Capacitismo', 'Xenofobia']
atrasados: ['Capacitismo', 'Xenofobia']
coxa: ['Capacitismo', 'Racismo']
esquisito: ['Capacitismo', 'Xenofobia']
louca: ['Misoginia', 'Capacitismo']
loucas: ['Misoginia', 'Capacitismo']
maluca: ['Misoginia', 'Capacitismo']
maluquinha: ['Misoginia', 'Capacitismo']
mudo: ['Capacitismo', 'Racismo']
anaguel: ['Misoginia', 'Gordofobia']
barriga: ['Gordofobia', 'Racismo']
gord*: ['Misoginia', 'Gordofobia']
gorda: ['Misoginia', 'Gordofobia']
gordas: ['Misoginia', 'Gordofobia']
gostosa: ['Misoginia', 'Gordofobia']
bugre: ['Xenofobia', 'Int.Religiosa']
deus: ['Prec.Politico', 'Int.Religiosa']
macumb*: ['Racismo', 'Int.Religiosa']
macumba: ['Racismo', 'Int.Religiosa']
oferenda*: ['Racismo', 'Int.Religiosa']
pecado*: ['Racismo', 'Int.Religiosa']
lésbica: ['Misoginia', 'LGBTFobia']
mariqui

#### Analisando quais palavras e quantas palavras do dicionário tem no PrejudiceWhatsApp.Br e PrejudiceTelegram.Br

In [4]:
import re
from collections import Counter

def load_dictionary_entries(dic_path):
    words = []
    with open(dic_path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            
            # Ignorar comentários e linhas vazias
            if not line or line.startswith("%"):
                continue
            
            parts = line.split("\t")
            
            # Ignorar linhas de definição de categorias (ex: "1\tCapacitismo")
            if parts[0].isdigit():
                continue
            
            words.append(parts[0])
    
    return words


entries = load_dictionary_entries(
    "./Dicionário/v2_SocialLIWC_formatado_ordenado.dic"
)

print(f"Total de entradas lexicais no dicionário: {len(entries)}")




Total de entradas lexicais no dicionário: 842


In [5]:
import pandas as pd

wpp_df = pd.read_csv("./Datasets/Correto_whatsapp_rotulado_revisado.csv")
telegram_df = pd.read_csv("./Datasets/Telegram_Tratado_Rotulado_revisado_Final.csv")

# Ajuste o nome da coluna se necessário
wpp_texts = wpp_df["text_content_anonymous"].astype(str)
telegram_texts = telegram_df["text_content_anonymous"].astype(str)


In [6]:
def extract_dictionary_words(texts, dic_words):
    found_words = Counter()
    
    for text in texts:
        tokens = re.findall(r"\b\w+\b", text.lower())
        for token in tokens:
            if token in dic_words:
                found_words[token] += 1
                
    return found_words

wpp_matches = extract_dictionary_words(wpp_texts, entries)
telegram_matches = extract_dictionary_words(telegram_texts, entries)

print(f"Palavras do dicionário encontradas no WhatsApp: {len(wpp_matches)}")
print(f"Palavras do dicionário encontradas no Telegram: {len(telegram_matches)}")


Palavras do dicionário encontradas no WhatsApp: 265
Palavras do dicionário encontradas no Telegram: 286


In [7]:
# Top 20 palavras mais frequentes
print("WhatsApp - Top 20:")
for word, freq in wpp_matches.most_common(20):
    print(word, freq)

print("\nTelegram - Top 20:")
for word, freq in telegram_matches.most_common(20):
    print(word, freq)


WhatsApp - Top 20:
deus 964
luladrão 343
ladrão 162
idiota 145
liberdade 99
burro 81
claro 77
estranho 71
viado 69
nordestino 68
imbecil 66
lulaladrão 61
esquerdista 57
homem 54
coronel 51
burra 47
coisas 43
bozo 42
mulher 38
inferno 37

Telegram - Top 20:
deus 1213
ladrão 380
luladrão 183
liberdade 123
claro 107
esquerdista 101
ditadura 89
terra 88
coisas 80
homem 74
esquerdalha 69
diabo 64
satanás 59
branco 57
inferno 53
ditador 53
estranho 49
justiça 47
santo 46
mulher 46
